### For Training 
You can run this notebook from end-to-end (You must uncommment out the training code (Not the GRPO code))
This assumes the following:
- kaggle.json is inside the root directory (during your first run)
- The "./sft-lora-adapter-3b" adapter exists as a directory in the root directory with its relevant files
- The "./sft-lora-adapter-3b-epoch2" adapter exists as a directory in the root directory with its relevant files

### For Inference 
You can run the notebook from the Inference Section until the end. 
This assumes the following:
- You have all the packages of requirements.txt installed
- The "./final-merged-epoch2" model exists as a directory in the root directory with its relevant files
- The directory "./dl-spring-2026-svg-generation" exists with all relevant files inside (train.csv, test.csv). This directory can be acquired by running cells 2 and 3.
- You have train_optimized.parquet inside your root directory (Run Cell #5 to obtain it)

In [ ]:
# Remember to make a venv and install the dependencies before running this notebook. You can do this with pip install -r requirements.txt if you have a requirements.txt file with the necessary dependencies, 
# py -m venv venv && source venv/bin/activate && pip install -r requirements.txt

# Or you can run the following commands to install them directly (after making your venv and activating it):

# !pip install git+https://github.com/unslothai/unsloth-zoo.git git+https://github.com/unslothai/unsloth.git pandas numpy torch pathlib kaggle trl==0.24.0 bitsandbytes picosvg cairosvg opencv-python-headless editdistance mergekit llm_blender outlines scikit-learn weave wandb protobuf llguidance

!pip install -r requirements.txt

  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-req-build-6pr8qh2a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth-zoo.git /tmp/pip-req-build-6pr8qh2a
  Resolved https://github.com/unslothai/unsloth-zoo.git to commit e5944f23827b49eda601e6168fa17def8338e96f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-av49yxvb
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-av49yxvb
  Resolved https://github.com/unslothai/unsloth.git to commit a6c1f893fc87c0973f9c32e59ca3d7d54ffb9724
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
'''
Assuming you plan to run cell 3, you will need to download your Kaggle API key and set it up in your environment. 
The following code snippet is a common way to do this in a Jupyter notebook. 
Make sure to replace 'kaggle.json' with the actual name of your API key file if it's different.
'''
# !mkdir -p ~/.kaggle
# !mv kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json

In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("./dl-spring-2026-svg-generation")

# Download data
if not BASE_DIR.exists() or not any(BASE_DIR.iterdir()):
    print("Downloading and unzipping data...")
    os.makedirs(BASE_DIR, exist_ok=True)
    print("Downloading...")
    !kaggle competitions download -c dl-spring-2026-svg-generation-from-text-prompts-extended-deadline --force
    print("Unzipping...")
    !unzip -q dl-spring-2026-svg-generation.zip
    !mv sample_submission.csv dl-spring-2026-svg-generation
    !mv train.csv dl-spring-2026-svg-generation
    !mv test.csv dl-spring-2026-svg-generation
else:
    print("SVG Data folder already exists.")

SVG Data folder already exists.


## Data Analysis Section

In [4]:
import pandas as pd
import re

print("Analyzing SVG Dimensions in the Training Set.")

# Load the dataset
df = pd.read_csv("./dl-spring-2026-svg-generation/train.csv")

# Regex to safely find attributes inside the opening <svg ... > tag
height_pattern = re.compile(r'<svg[^>]*\sheight="([^"]+)"')
width_pattern = re.compile(r'<svg[^>]*\swidth="([^"]+)"')
viewbox_pattern = re.compile(r'<svg[^>]*\sviewBox="([^"]+)"')

heights = []
widths = []
viewboxes = []

# Scan the dataset
for svg in df['svg'].dropna():
    # We only search the first 250 characters to save time (the opening tag)
    head = svg[:250] 
    
    h_match = height_pattern.search(head)
    w_match = width_pattern.search(head)
    vb_match = viewbox_pattern.search(head)
    
    if h_match: heights.append(h_match.group(1))
    if w_match: widths.append(w_match.group(1))
    if vb_match: viewboxes.append(vb_match.group(1))

# Print the Results
print("\n--- Top 5 Heights ---")
print(pd.Series(heights).value_counts().head(5).to_string())

print("\n--- Top 5 Widths ---")
print(pd.Series(widths).value_counts().head(5).to_string())

print("\n--- Top 5 ViewBoxes ---")
print(pd.Series(viewboxes).value_counts().head(5).to_string())

total_svgs = len(df)
print(f"\nTotal SVGs scanned: {total_svgs}")

Analyzing SVG Dimensions in the Training Set.

--- Top 5 Heights ---
200.0px    40892
128         6308
400          740
200          394
100          245

--- Top 5 Widths ---
200.0px    40892
128         6308
400          733
200          384
100          245

--- Top 5 ViewBoxes ---
0.0 0.0 200.0 200.0    40892
0 0 24 24               2104
0 0 128 128              902
0 0 400 400              734
0 0 32 32                655

Total SVGs scanned: 50000


In [ ]:
import pandas as pd
import numpy as np
import concurrent.futures
import re
import os
from picosvg.svg import SVG

# Regex Compilations
FLOAT_PATTERN = re.compile(r'-?\d+\.\d+')
PATH_LETTER_SPACE_PATTERN = re.compile(r'([a-zA-Z])\s+')
COMMA_SPACE_PATTERN = re.compile(r'\s*,\s*')
HEX_COLOR_PATTERN = re.compile(r'#([0-9a-fA-F])\1([0-9a-fA-F])\2([0-9a-fA-F])\3')
MULTIPLE_SPACES_PATTERN = re.compile(r'\s+')
PATH_D_ATTR_PATTERN = re.compile(r'd="([^"]+)"')

# Cleaning Helper Functions 

def round_floats(match):
    # Strictly maintains 2 decimal points, but strips useless trailing zeros
    return f"{float(match.group(0)):.2f}".rstrip('0').rstrip('.')

def minify_d_attr(match):
    """Cleans spaces strictly inside the path string."""
    d_str = match.group(1)
    # Remove spaces after path command letters (e.g., "M 10" -> "M10")
    d_str = PATH_LETTER_SPACE_PATTERN.sub(r'\1', d_str)
    # Remove spaces around commas (e.g., "10, 20" -> "10,20")
    d_str = COMMA_SPACE_PATTERN.sub(',', d_str)
    # Condense multiple spaces into one
    d_str = MULTIPLE_SPACES_PATTERN.sub(' ', d_str)
    return f'd="{d_str}"'

def minify_path_and_hex(svg_string):
    # Apply the aggressive space removal only to the d="..." attribute
    svg_string = PATH_D_ATTR_PATTERN.sub(minify_d_attr, svg_string)
    
    # Compress hex colors globally (e.g., #FFFFFF -> #FFF)
    svg_string = HEX_COLOR_PATTERN.sub(r'#\1\2\3', svg_string)
    
    # Condense global multiple spaces safely
    svg_string = MULTIPLE_SPACES_PATTERN.sub(' ', svg_string)
    
    return svg_string

def strip_useless_tags(svg_string):
    # Remove empty defs
    svg_string = svg_string.replace("<defs/>", "").replace("<defs></defs>", "")
    
    # Remove redundant/default attributes to save tokens
    svg_string = re.sub(r'\s*fill-opacity="1\.0"', '', svg_string)
    svg_string = re.sub(r'\s*filling="0"', '', svg_string)
    svg_string = re.sub(r'\s*fill=""', '', svg_string)
    
    # Normalize height and width instead of deleting them
    # This changes height="200.0px" or width="256px" to simply height="200"
    svg_string = re.sub(r'\s*height="[0-9.]+(px)?"', ' height="200"', svg_string)
    svg_string = re.sub(r'\s*width="[0-9.]+(px)?"', ' width="200"', svg_string)
    
    # Close gaps between XML tags
    svg_string = svg_string.replace('> <', '><')
    
    return svg_string

# Main Optimization Wrapper 
def optimize_svg(raw_svg):
    if not isinstance(raw_svg, str) or not raw_svg.strip(): 
        return ""
        
    try:
        # Step A: Standardize shapes to paths using picosvg
        svg_obj = SVG.fromstring(raw_svg)
        cleaned_svg = svg_obj.topicosvg().tostring()
        
        # Step B: Apply 2-decimal float rounding
        cleaned_svg = FLOAT_PATTERN.sub(round_floats, cleaned_svg)
        
        # Step C: Minify paths and compress hex codes
        cleaned_svg = minify_path_and_hex(cleaned_svg)
        
        # Step D: Strip useless XML bloat
        cleaned_svg = strip_useless_tags(cleaned_svg)
        
        return cleaned_svg.strip()
    except Exception:
        return ""

# Parallel Processing Execution 
def process_chunk(chunk):
    chunk['svg'] = chunk['svg'].apply(optimize_svg)
    return chunk
    
# This process took 20 minutes with 4 cores (Comment out the below lines if you already have the train_optimized.parquet file)
# if __name__ == "__main__":
#     print("Starting Data Diet.")
    
#     df = pd.read_csv("./dl-spring-2026-svg-generation/train.csv")
    
#     num_cores = os.cpu_count() or 4
#     num_cores = 4
#     print(f"At least {num_cores} Cores are available.")

#     indices = np.array_split(df.index, num_cores * 4)
#     chunks = [df.loc[idx].copy() for idx in indices]

#     processed_chunks = []
#     with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
#         for result in executor.map(process_chunk, chunks):
#             processed_chunks = list(executor.map(process_chunk, chunks))

#     final_df = pd.concat(processed_chunks)
    
#     # Filter out any SVGs that failed parsing or returned empty
#     final_df = final_df[final_df['svg'] != ""]
    
#     final_df.to_parquet("./train_optimized.parquet", index=False)
#     print(f"Data Diet Complete. {len(final_df)} highly compressed rows saved.")

In [ ]:
# Load the newly compressed dataset
df = pd.read_parquet("./train_optimized.parquet")

print(f"Total optimized rows ready for training: {len(df)}\n")
print("==================================================")
print("       FIRST 10 SVGS FOR SANITY CHECK")
print("==================================================\n")

# Loop through the first 10 rows
for i, row in df.head(10).iterrows():
    print(f"[{i+1}/10] PROMPT: {row['prompt']}")
    print(f"SVG STRING:\n{row['svg']}\n")
    print("-" * 80 + "\n")

Total optimized rows ready for training: 49043

       FIRST 10 SVGS FOR SANITY CHECK

[1/10] PROMPT: The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background.
SVG STRING:
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200" width="200"><path fill="#FF6A00" d="M93.3,21.2 L93.3,80.4 L21.2,80.4 L21.2,179.6 L120.4,179.6 L120.4,107.1 L179.1,107.1 L179.1,21.2 L93.3,21.2 ZM113.8,172.9 L27.9,172.9 L27.9,87.1 L113.7,87.1 L113.7,172.9 L113.8,172.9 ZM172.5,100.4 L120.4,100.4 L120.4,80.4 L100,80.4 L100,27.9 L172.5,27.9 L172.5,100.4 Z"/><path fill="#FF6A00" d="M143.8,44.2 C143.8,40 140.5,37.1 136.7,37.1 C132.5,37.1 129.6,40.4 129.6,44.2 L129.6,51.3 L144.2,51.3 L144.2,44.2 L143.8,44.2 ZM129.2,62.1 C129.2,66.3 132.5,69.2 136.3,69.2 C140.5,69.2 143.4,65.9 143.4,62.1 L143.4,55 L129.2,55 L129.2,62.1 Z"/><path fill="#FF6A00" d="M149.6,62.1 C149.6,68.8 144.6,74.2 138.4,75 L135,75 C128.8,74.2 123.8,68.8 123.8,62.1 L116.

## Training Section

### Epoch #1 

In [ ]:
# import os
# import torch
# from unsloth import FastLanguageModel

# import transformers.utils.hub
# transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", "~/.cache/huggingface/hub")
# os.environ["WANDB_DISABLED"] = "true"

# from trl import SFTTrainer, SFTConfig
# from datasets import load_dataset

# if __name__ == "__main__":
#     train_optimized_filepath = "./train_optimized.parquet"
#     raw_dataset = load_dataset("parquet", data_files=train_optimized_filepath, split="train")
    
#     # Split into Train and Eval 
#     dataset_split = raw_dataset.select(range(25000)).train_test_split(test_size=500, seed=25)
#     train_dataset = dataset_split["train"]
#     eval_dataset = dataset_split["test"]
    
#     MODEL_ID = "unsloth/Qwen2.5-Coder-3B"
    
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name=MODEL_ID,
#         max_seq_length=2048,
#         dtype=torch.float16, 
#         load_in_4bit=True,
#     )
    
#     model = FastLanguageModel.get_peft_model(
#         model, 
#         # Increased Capacity & MLP Layers 
#         r=32, 
#         lora_alpha=64, 
#         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
#         lora_dropout=0, 
#         bias="none",
#         use_gradient_checkpointing=True,
#         random_state=25,
#     )
    
#     EOS = tokenizer.eos_token
    
#     # Format the dataset for SFT by combining prompt and optimized SVG into a single text field
#     def format_for_sft(example):
#         open_comment = "<!-" + "-"
#         close_comment = "-" + "->"
#         text = f"{open_comment} Description: {example['prompt']} {close_comment}\n```xml\n{example['svg']}\n```{EOS}"
#         return {"text": text}
    
#     train_dataset = train_dataset.map(format_for_sft)
#     eval_dataset = eval_dataset.map(format_for_sft)
    
#     sft_config = SFTConfig(
#         output_dir="./svg-phase1-sft",
#         dataset_text_field="text",
#         max_seq_length=2048,
        
#         # Packing Enabled to Squeeze More Learning Signal
#         packing=True, 
        
#         num_train_epochs=1, 
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=8, 
        
#         # Advanced Training Dynamics 
#         learning_rate=2e-4,
#         lr_scheduler_type="cosine", 
#         warmup_steps=100,           
#         weight_decay=0.01,          
#         neftune_noise_alpha=5,      
        
#         fp16=True,  
#         bf16=False, 
#         optim="paged_adamw_8bit", 
#         logging_steps=50,
        
#         # Evaluation Strategy 
#         eval_strategy="steps",
#         eval_steps=200, 
        
#         # Saves exactly when you evaluate, and keeps the best one
#         save_strategy="steps", 
#         save_steps=200,
#         load_best_model_at_end=True,         # Auto-reverts to the best checkpoint!
#         metric_for_best_model="eval_loss",   # Uses eval set as the judge
#         greater_is_better=False,             # Because lower loss is better
#         save_total_limit=2,
#         ddp_find_unused_parameters=False,
#     )
    
#     sft_trainer = SFTTrainer(
#         model=model, 
#         tokenizer=tokenizer, 
#         train_dataset=train_dataset, 
#         eval_dataset=eval_dataset, 
#         args=sft_config
#     )
    
#     print("Starting SFT Phase 1 with Qwen2.5-Coder-3B...")
#     sft_trainer.train()
    
#     model.save_pretrained("./sft-lora-adapter-3B")
#     tokenizer.save_pretrained("./sft-lora-adapter-3B")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading base model to CPU.")
# Load the base model strictly to the CPU to avoid the 8GB VRAM limit
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-Coder-3B", 
    torch_dtype=torch.float16,
    device_map="cpu", 
)

print("Attaching LoRA adapter.")
# Attach your Phase 1 adapter
model = PeftModel.from_pretrained(
    base_model,
    "./sft-lora-adapter-3B",
    device_map="cpu",
)

print("Merging weights (this will take a while).")
# Fuse the LoRA weights directly into the base model's linear layers
merged_model = model.merge_and_unload()

print("Saving merged model to disk.")
# Export the standalone Hugging Face model
merged_model.save_pretrained("./sft-merged-model-3b")

tokenizer = AutoTokenizer.from_pretrained("./sft-lora-adapter-3B")
tokenizer.save_pretrained("./sft-merged-model-3b")

print("Merge complete! Safe for Outlines inference/GRPO.")

/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


Loading base model to CPU.


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:31<00:00, 13.69it/s]


Attaching LoRA adapter.
Merging weights (this will take a while).
Saving merged model to disk.


Writing model shards: 100%|██████████| 1/1 [01:53<00:00, 113.43s/it]


Merge complete! Safe for Outlines inference/GRPO.


In [9]:
import gc
import torch

del model, base_model, merged_model, tokenizer
gc.collect()
torch.cuda.empty_cache()

## Epoch #2

In [ ]:
# import os
# import torch
# from unsloth import FastLanguageModel

# import transformers.utils.hub
# transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", "~/.cache/huggingface/hub")
# os.environ["WANDB_DISABLED"] = "true"

# from trl import SFTTrainer, SFTConfig
# from datasets import load_dataset

# if __name__ == "__main__":
#     train_optimized_filepath = "./train_optimized.parquet"
#     raw_dataset = load_dataset("parquet", data_files=train_optimized_filepath, split="train")
    
#     # THE DATA ALPHA STRATEGY 
#     # Instead of repeating the first 25,000, we feed it the REMAINING 24,043 SVGs!
#     # The model learns entirely new concepts it has never seen before.
#     dataset_split = raw_dataset.select(range(25000, 49043)).train_test_split(test_size=500, seed=42)
#     train_dataset = dataset_split["train"]
#     eval_dataset = dataset_split["test"]
    
#     # Load the 16-bit merged model 
#     MODEL_ID = "./sft-merged-model-3b" 
    
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name=MODEL_ID,
#         max_seq_length=2048,
#         dtype=torch.float16, 
#         load_in_4bit=True,
#     )
    
#     # We apply a fresh LoRA adapter to capture these new, refined details
#     model = FastLanguageModel.get_peft_model(
#         model, 
#         r=32, 
#         lora_alpha=64, 
#         target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], 
#         lora_dropout=0, 
#         bias="none",
#         use_gradient_checkpointing=True,
#         random_state=42, # Changed seed to shake up the data ordering
#     )
    
#     EOS = tokenizer.eos_token
    
#     # Format the dataset for SFT by combining prompt and optimized SVG into a single text field
#     def format_for_sft(example):
#         open_comment = "<!-" + "-"
#         close_comment = "-" + "->"
#         text = f"{open_comment} Description: {example['prompt']} {close_comment}\n```xml\n{example['svg']}\n```{EOS}"
#         return {"text": text}
    
#     train_dataset = train_dataset.map(format_for_sft)
#     eval_dataset = eval_dataset.map(format_for_sft)
    
#     sft_config = SFTConfig(
#         output_dir="./svg-phase2-sft-epoch2", # Save to a new folder!
#         dataset_text_field="text",
#         max_seq_length=2048,
        
#         # Packing Enabled to Squeeze More Learning Signal
#         packing=True, 
        
#         num_train_epochs=1, 
#         per_device_train_batch_size=1,
#         gradient_accumulation_steps=8, 
        
#         # THE WARM RESTART DYNAMICS
#         learning_rate=5e-5,          # Dropped from 2e-4 to gently refine the weights
#         lr_scheduler_type="cosine", 
#         warmup_ratio=0.05,           # 5% warmup for the new curve
#         weight_decay=0.01,          
#         neftune_noise_alpha=5,      
        
#         fp16=True,  
#         bf16=False, 
#         optim="paged_adamw_8bit", 
#         logging_steps=50,
        
#         eval_strategy="steps",
#         eval_steps=200, 
        
#         save_strategy="steps", 
#         save_steps=200,
#         load_best_model_at_end=True,         
#         metric_for_best_model="eval_loss",   
#         greater_is_better=False,             
#         save_total_limit=2,
#         ddp_find_unused_parameters=False,
#     )
    
#     sft_trainer = SFTTrainer(
#         model=model, 
#         tokenizer=tokenizer, 
#         train_dataset=train_dataset, 
#         eval_dataset=eval_dataset,
#         args=sft_config
#     )
    
#     print("Starting SFT Epoch 2 (Part 2) with Merged Qwen2.5-Coder-3B...")
#     sft_trainer.train()
    
#     # Save the new adapter!
#     model.save_pretrained("./sft-lora-adapter-3B-epoch2")
#     tokenizer.save_pretrained("./sft-lora-adapter-3B-epoch2")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Setup your paths
# Point this to the exact base model used for Epoch 2
BASE_MODEL_DIR = "./sft-merged-model-3b" 
ADAPTER_DIR = "./sft-lora-adapter-3B-epoch2"
OUTPUT_DIR = "./final-merged-epoch2"

# Load the Base Model in 16-bit Float
print(f"Loading Base Model from {BASE_MODEL_DIR}...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True, # Helps prevent RAM spikes during loading
    device_map="cpu"        # Safest to do the merge entirely in CPU RAM
)

print(f"Loading Tokenizer from {ADAPTER_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

# Apply the LoRA Adapter to the Base Model
print(f"Applying LoRA Adapter from {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

# Bake the weights together
print("Merging LoRA weights into base model layers (this takes a minute)...")
merged_model = model.merge_and_unload()

# Save the final model
print(f"Saving fully merged 16-bit model to {OUTPUT_DIR}...")
merged_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Merge complete! Ready for Inference.")

/mnt/c/Users/Josue/Documents/CS-GY-6923- Deep Learning/Midterm/unsloth_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
`torch_dtype` is deprecated! Use `dtype` instead!


Loading Base Model from ./sft-merged-model-3b...


Loading weights: 100%|██████████| 434/434 [00:00<00:00, 1402.12it/s]


Loading Tokenizer from ./sft-lora-adapter-3B-epoch2...
Applying LoRA Adapter from ./sft-lora-adapter-3B-epoch2...
Merging LoRA weights into base model layers (this takes a minute)...
Saving fully merged 16-bit model to ./final-merged-epoch2...


Writing model shards: 100%|██████████| 1/1 [01:34<00:00, 94.87s/it]


Merge complete! Ready for Inference.


In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Load your local SFT-trained tokenizer
# You can point this to "./final-merged-epoch2" or "Qwen/Qwen2.5-Coder-3B"
TOKENIZER_PATH = "./final-merged-epoch2" 

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# Load the highly optimized dataset
PARQUET_PATH = "./train_optimized.parquet"
print(f"Loading {PARQUET_PATH}...")
df = pd.read_parquet(PARQUET_PATH)

print("Calculating exact token lengths for 49,043 SVGs (this takes about 60 seconds)...")

# Encode the SVGs and count the tokens
# We add_special_tokens=False because we only care about the SVG payload length
lengths = df['svg'].apply(lambda x: len(tokenizer.encode(str(x), add_special_tokens=False)))

# Extract the critical metrics
max_len = lengths.max()
p99_len = lengths.quantile(0.99)
p95_len = lengths.quantile(0.95)
avg_len = lengths.mean()

print("\n======================================")
print("       SVG TOKEN LENGTH STATS         ")
print("======================================")
print(f"Absolute Max Length: {max_len} tokens")
print(f"99th Percentile:     {p99_len:.0f} tokens")
print(f"95th Percentile:     {p95_len:.0f} tokens")
print(f"Average Length:      {avg_len:.0f} tokens")
print("======================================")

Loading tokenizer...
Loading ./train_optimized.parquet...
Calculating exact token lengths for 49,043 SVGs (this takes about 60 seconds)...


Token indices sequence length is longer than the specified maximum sequence length for this model (33864 > 32768). Running this sequence through the model will result in indexing errors



       SVG TOKEN LENGTH STATS         
Absolute Max Length: 44909 tokens
99th Percentile:     4988 tokens
95th Percentile:     2503 tokens
Average Length:      1173 tokens


In [3]:
import gc
import torch

del model, base_model, merged_model, tokenizer
gc.collect()
torch.cuda.empty_cache()

## Inference Section

Assuming you have the model's weights in your root directory ("./final-merged-epoch2"), you can start the notebook from here. 

(This also assumes you pip installed the required packages/libraries beforehand)

In [ ]:
import os
import pandas as pd
import torch
import io
import cairosvg
import numpy as np
from PIL import Image
import xml.etree.ElementTree as ET
import warnings
import re

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel
import outlines
from outlines.types import CFG

warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

print("Initializing Local Inference Process...")
device = "cuda"

# ==========================================
# LOAD MAIN MODEL (16-BIT PRECISION)
# ==========================================
print("Loading 3B model in 16-bit (Float16).")
hf_model = AutoModelForCausalLM.from_pretrained(
    "./final-merged-epoch2",
    torch_dtype=torch.float16, # This replaces the 4-bit quantization!
    device_map=device, 
)

hf_model.generation_config.max_length = None
hf_tokenizer = AutoTokenizer.from_pretrained("./final-merged-epoch2")

# The Outlines CFG guarantees 100% valid SVG formatting
generator = outlines.from_transformers(hf_model, hf_tokenizer)

official_kaggle_grammar = CFG("""
    ?start: WS? svg WS?
    svg: "<svg" WS? ATTR_LIST ">" WS? elements "</svg>"
    elements: (element | TEXT)*
    element: "<" TAG WS? ATTR_LIST "/>" WS? | "<" TAG WS? ATTR_LIST ">" WS? elements "</" TAG ">" WS?
    TAG: "svg" | "g" | "path" | "rect" | "circle" | "ellipse" | "line" | "polyline" | "polygon" | "defs" | "use" | "symbol" | "clipPath" | "mask" | "linearGradient" | "radialGradient" | "stop" | "text" | "tspan" | "title" | "desc" | "style" | "pattern" | "marker" | "filter"
    ATTR_LIST: (/[a-zA-Z0-9_:-]+/ WS? "=" WS? /"[^"]*"/ WS?)*
    TEXT: /[^<]+/
    WS: /[ \\t\\n\\r]+/
""")

# ==========================================
# LOAD SIGLIP JUDGE (16-BIT PRECISION)
# ==========================================
print("Loading SigLIP Judge.")
siglip_id = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(siglip_id)

judge = AutoModel.from_pretrained(
    siglip_id, 
    torch_dtype=torch.float16 # Crucial for saving VRAM
).to(device)
judge.eval() 


Initializing local inference.
Loading 3B model in 16-bit (Float16).


Loading weights: 100%|██████████| 434/434 [00:28<00:00, 15.29it/s]


Loading SigLIP Judge.


The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 888/888 [00:20<00:00, 43.40it/s]


SiglipModel(
  (text_model): SiglipTextTransformer(
    (embeddings): SiglipTextEmbeddings(
      (token_embedding): Embedding(32000, 1152)
      (position_embedding): Embedding(64, 1152)
    )
    (encoder): SiglipEncoder(
      (layers): ModuleList(
        (0-26): 27 x SiglipEncoderLayer(
          (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (self_attn): SiglipAttention(
            (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
            (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (mlp): SiglipMLP(
            (activation_fn): GELUTanh()
            (fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (fc2): Linear(in_features=4304,

In [ ]:
# ==========================================
# HELPER FUNCTIONS
# ==========================================
def extract_svg(text):
    match = re.search(r'(<svg.*?</svg>)', text, re.IGNORECASE | re.DOTALL)
    return match.group(1) if match else text

def render_to_numpy(svg_string):
    try:
        png_data = cairosvg.svg2png(bytestring=svg_string.encode('utf-8'), output_width=256, output_height=256, background_color="white")
        return np.array(Image.open(io.BytesIO(png_data)).convert('L'))
    except: return None

def is_kaggle_compliant(svg_string):
    if len(svg_string) > 16000: return False
    try:
        root = ET.fromstring(svg_string)
        if root.tag.split('}')[-1] != 'svg': return False
    except ET.ParseError: return False
    if svg_string.count("<path") > 256: return False
    return True

def heal_svg(raw_svg):
    """Saves truncated SVGs by chopping off broken paths and closing the tag."""
    if raw_svg.strip().endswith("</svg>"):
        return raw_svg
        
    # Find the last perfectly closed element (like a completed <path />)
    last_closed_idx = raw_svg.rfind("/>")
    
    if last_closed_idx != -1:
        # Chop off the broken half-written path and seal the SVG
        healed_svg = raw_svg[:last_closed_idx + 2] + "\n</svg>"
        return healed_svg
        
    return raw_svg

def select_best_svg(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = extract_svg(raw_svg)

        # INVESTIGATION PRINT
        if not is_kaggle_compliant(clean_svg):
            print(f"      [!] Candidate {i+1} failed compliance. Length: {len(clean_svg)}")
            print(f"Raw string: {clean_svg[-100:]}") # Uncomment to see how it ended
            continue
            
        img = render_to_numpy(clean_svg)
        if img is None:
            print(f"      [!] Candidate {i+1} failed CairoSVG render.")
            continue
            
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(clean_svg) 
            
    if not valid_images: 
        print("      [X] ALL CANDIDATES FAILED. Returning empty fallback.")
        return "<svg></svg>"
        
    if len(valid_images) == 1: return valid_svgs[0]

    # Prepare the inputs for the judge model
    inputs = processor(
        text=[prompt_text], 
        images=valid_images, 
        return_tensors="pt", 
        padding="max_length", 
        truncation=True, 
        max_length=64    
    ).to(device)
    
    # Run the judge model to get similarity scores
    with torch.no_grad():
        outputs = judge(**inputs)
        scores = outputs.logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0:
        return valid_svgs[0]
        
    return valid_svgs[scores.argmax()]

def select_best_svg_healed(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = heal_svg(extract_svg(raw_svg))

        # INVESTIGATION PRINT
        if not is_kaggle_compliant(clean_svg):
            print(f"      [!] Candidate {i+1} failed compliance. Length: {len(clean_svg)}")
            print(f"Raw string: {clean_svg[-100:]}") # Uncomment to see how it ended
            continue
            
        img = render_to_numpy(clean_svg)
        if img is None:
            print(f"      [!] Candidate {i+1} failed CairoSVG render.")
            continue
            
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(clean_svg) 
            
    if not valid_images: 
        print("      [X] ALL CANDIDATES FAILED. Returning empty fallback.")
        return "<svg></svg>"
        
    if len(valid_images) == 1: return valid_svgs[0]

    inputs = processor(
        text=[prompt_text], 
        images=valid_images, 
        return_tensors="pt", 
        padding="max_length", 
        truncation=True, 
        max_length=64    
    ).to(device)
    
    with torch.no_grad():
        outputs = judge(**inputs)
        scores = outputs.logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0:
        return valid_svgs[0]
        
    return valid_svgs[scores.argmax()]


# Updated is_kaggle_compliant and Heal Functions with better geometry checks and quote-aware tag healing
def is_kaggle_compliant_updated(svg_string):
    # Length Check
    if len(svg_string) > 16000: 
        return False
        
    # Geometry Check
    # An SVG must contain at least one of these tags to be considered "drawn"
    drawing_tags = [
        '<path', '<rect', '<circle', '<ellipse', 
        '<line', '<polyline', '<polygon', '<text', '<use'
    ]
    if not any(tag in svg_string for tag in drawing_tags):
        print("      [!] Candidate failed: No geometric shapes found.")
        return False

    # XML Parsing Check
    try:
        root = ET.fromstring(svg_string)
        if root.tag.split('}')[-1] != 'svg': 
            return False
    except ET.ParseError: 
        return False
        
    # Kaggle Path Limit Check
    if svg_string.count("<path") > 256: 
        return False
        
    return True
    
def heal_svg_updated(raw_svg):
    """
    Intelligently seals truncated SVGs by tracking unclosed tags 
    via a LIFO stack, completely ignoring '>' symbols inside attributes.
    """
    raw_svg = raw_svg.strip()
    if raw_svg.endswith("</svg>"): 
        return raw_svg
        
    # Clean up the trailing broken text
    # We use a custom rfind that respects quotes to find the true last tag closure
    in_quotes = False
    last_closed_idx = -1
    for i, char in enumerate(raw_svg):
        if char == '"':
            in_quotes = not in_quotes
        elif char == '>' and not in_quotes:
            last_closed_idx = i
            
    if last_closed_idx != -1:
        raw_svg = raw_svg[:last_closed_idx + 1]
        
    # Track open tags using a quote-aware regex
    stack = []
    
    # UPGRADE: Match `<` then either (non-quote-non-bracket) OR ("quoted string") until `>`
    tags = re.findall(r'<((?:[^>"]|"[^"]*")*)>', raw_svg)
    
    for tag in tags:
        tag = tag.strip()
        
        if tag.startswith('/'):
            tag_name = tag[1:].strip()
            if stack and stack[-1] == tag_name:
                stack.pop()
                
        elif tag.endswith('/'):
            continue # Self-closing tag, safe!
            
        elif tag.startswith('?') or tag.startswith('!'):
            continue # Comments/XML declarations, safe!
            
        else:
            tag_name = tag.split()[0]
            stack.append(tag_name)
            
    # Close remaining tags (LIFO)
    for tag_name in reversed(stack):
        if tag_name.lower() != 'svg': 
            raw_svg += f"\n</{tag_name}>"
            
    if not raw_svg.strip().endswith("</svg>"):
        raw_svg += "\n</svg>"
        
    return raw_svg

def select_best_svg_healed_updated(prompt_text, candidate_svgs):
    valid_images, valid_svgs = [], []
    
    for i, raw_svg in enumerate(candidate_svgs):
        clean_svg = heal_svg_updated(extract_svg(raw_svg))

        # INVESTIGATION PRINT
        if not is_kaggle_compliant_updated(clean_svg):
            print(f"      [!] Candidate {i+1} failed compliance. Length: {len(clean_svg)}")
            print(f"Raw string: {clean_svg[-100:]}") # Uncomment to see how it ended
            continue
            
        img = render_to_numpy(clean_svg)
        if img is None:
            print(f"      [!] Candidate {i+1} failed CairoSVG render.")
            continue
            
        valid_images.append(Image.fromarray(img).convert('RGB'))
        valid_svgs.append(clean_svg) 
            
    if not valid_images: 
        print("      [X] ALL CANDIDATES FAILED. Returning empty fallback.")
        return "<svg></svg>"
        
    if len(valid_images) == 1: return valid_svgs[0]

    inputs = processor(
        text=[prompt_text], 
        images=valid_images, 
        return_tensors="pt", 
        padding="max_length", 
        truncation=True, 
        max_length=64    
    ).to(device)
    
    with torch.no_grad():
        outputs = judge(**inputs)
        scores = outputs.logits_per_image.squeeze().cpu().numpy()
        
    if scores.ndim == 0:
        return valid_svgs[0]
        
    return valid_svgs[scores.argmax()]

The below process took 1477 Minutes to train for 1000 samples.

In [ ]:
# ==========================================
# INFERENCE LOOP
# ==========================================
if __name__ == "__main__":
    test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv") 
    csv_filename = "./submission-3b-2048-p2.csv"
    resume_index = 0
    
    # We will generate 2 candidates per prompt to feed to SigLIP
    NUM_CANDIDATES = 2
    
    # We assume that the Max Safe Context window for our model after fine-tuning is 2048 tokens (As our starting point), 
    # so we must ensure prompt + generation fits within this limit
    MAX_CONTEXT_WINDOW = 2048

    print(f"\nStarting Inference on {len(test_df)} prompts...")

    for idx in range(resume_index, len(test_df)):
        row = test_df.iloc[idx]
        print(f"\n[{idx+1}/{len(test_df)}] Generating ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        open_comment = "<!-" + "-"
        close_comment = "-" + "->"
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        
        # Calculate the token count of the prompt to ensure we stay within the model's context window
        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        candidates = [] 
        
        # GENERATION LOOP 
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # We ask outlines to generate exactly NUM_CANDIDATES sequence at a time
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
                candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        best_svg = select_best_svg(row["prompt"], candidates)
        print(best_svg[:200])  # Print the first 200 chars of the best SVG for sanity check
        
        result_df = pd.DataFrame([{"id": row["id"], "svg": best_svg}])
        if idx == 0 or not os.path.exists(csv_filename):
            result_df.to_csv(csv_filename, index=False, mode='w')
        else:
            result_df.to_csv(csv_filename, index=False, mode='a', header=False)
            
        # Clear cache before moving to the next totally new prompt
        torch.cuda.empty_cache()

    print("\nLocal inference complete! Check submission-3b-2048-p2.csv")


Starting Inference on 1000 prompts...

[1/1000] Generating ID: fa1d8fa7-080f-4269-a9cf-a17562c9a0ca | Prompt: firewood stack cut logs wood with leaf i...
  ~ Prompt tokens: 17 | Tokens available for SVG: 2021
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 1 failed compliance. Length: 3160
Raw string: ,220 Z"/><path fill="#2ca9bc" d="M2,222 L10,222 L10,224 L2,224 Z"/><path fill="#2ca9bc" d="M2,226 L1
<svg xmlns="http://www.w3.org/2000/svg" height="200" viewBox="0 0 24 24" width="200"><path fill="#000" d="M1.25,7.25 L2.75,7.25 L2.75,14.25 L1.25,14.25 ZM2.75,7.25 L2.75,14.25 L1.25,14.25 ZM1.25,7.25 

[2/1000] Generating ID: 6eede943219547c22ac56085027d33cc | Prompt: The image shows five horizontal lines of...
  ~ Prompt tokens: 27 | Tokens available for SVG: 2011
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200" width="200"><pat

The below process took 1749 minutes to train with 369 samples

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from the initial submission
sub_df = pd.read_csv("./submission-3b-2048-p2.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since it's our second pass
MAX_CONTEXT_WINDOW = 4096  # We can also try to push this up since we know that these cases are harder

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found! Saving the original submission as rescued.")
    sub_df.to_csv("./submission_rescued-3b-2048-p2.csv", index=False) 
else:
    # The 4096-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP 
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: MAX_CONTEXT_WINDOW bumped to 4096 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the robust select_best_svg_healed function here which includes the improved healing and compliance checks before scoring
        # (It already includes the is_kaggle_compliant check and CLIP scoring)
        best_svg = select_best_svg_healed(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the final rescue file
        sub_df.to_csv("./submission_rescued-3b-2048-p2.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_rescued-3b-2048-p2.csv' is ready.")

Found 369 blank SVGs. Starting Rescue Run.

[1/369] Rescuing ID: 9e831fb6831745f4d15156b7a95e4f92 | Prompt: An orange hexagon with the number '13' i...
  ~ Prompt tokens: 21 | Tokens available for SVG: 4065
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 2 failed compliance. Length: 4274
Raw string: .05 C54.08,151.04 50,148.76 46.16,145.76 L46.16,144.94 C32.58,130.35 25.04,120.37 25.04,112.27 C25.0
132.01,178.01 L132.01,183.04 C132.01,183.04 100.01,183.17 100,183.17 L94.94,183.17 C53.16,183.17 21.16,183.04 21.16,183.04 L21.16,178.01 C21.16,145.13 53.16,102.85 53.16,102.85 L100,102.85 Z"/>
</svg>

[2/369] Rescuing ID: 8ba1bd7c-211c-43b0-a4aa-0e6e33c92486 | Prompt: A pair of men's underwear with a contour...
  ~ Prompt tokens: 30 | Tokens available for SVG: 4056
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate 2 failed compliance. Length: 4231
Raw string: 09.9 C14.8,210.97 14.13,211

In [ ]:
# Identify the remaining blank SVGs
missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 73 blank SVGs.


The below process took 484 minutes to train with 73 samples.

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from the rescue submission
sub_df = pd.read_csv("./submission_rescued-3b-2048-p2.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since its our third pass
MAX_CONTEXT_WINDOW = 5050  # We can also try to push this up again since we know that these cases are even harder.

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found! Saving the previous submission as fixed.")
    sub_df.to_csv("./submission_fixed-3b-2048-p2.csv", index=False) 
else:
    # The 5050-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # GENERATION LOOP
        print(f"  ~ Generating {NUM_CANDIDATES} Candidates...")
        try:
            # THE KEY DIFFERENCE: max_new_tokens bumped to 5100 to prevent the guillotine
            raw_candidate = generator(
                prompt_text, 
                official_kaggle_grammar, 
                do_sample=True, 
                temperature=0.7, 
                max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                num_return_sequences=NUM_CANDIDATES, 
            )
            
            # Outlines might return a list of 1 string or just a string, handle both safely
            if isinstance(raw_candidate, list):
               candidates = raw_candidate
            else:
                candidates.append(raw_candidate)
                
        except Exception as e:
            print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the select_best_svg_healed_updated function, which includes further improved healing and compliance checks before scoring.
        # (It already includes the is_kaggle_compliant_updated check and CLIP scoring)
        best_svg = select_best_svg_healed_updated(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the fixed file
        sub_df.to_csv("./submission_fixed-3b-2048-p2.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_fixed-3b-2048-p2.csv' is ready.")

Found 73 blank SVGs. Starting Rescue Run.

[1/73] Rescuing ID: 4e56e508270eb0d02291927638d3c685 | Prompt: A single, dark gray outline of an abstra...
  ~ Prompt tokens: 27 | Tokens available for SVG: 5013
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate failed: No geometric shapes found.
      [!] Candidate 2 failed compliance. Length: 94
Raw string: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200" width="200">
</svg>
53.98 L164.02,50.71 A20.99 20.99 0 0 1 182.97,64.4 L185.23,70.67 A33.73 33.73 0 0 1 169.94,88.52 L176.22,91.79 A20.99 20.99 0 0 1 159.53,109.71 L166.8,112.98 A33.73 33.73 0 0 1 137.09,97.04 Z"/></svg>

[2/73] Rescuing ID: 9bda86e2078baf4d4fa15be8de3bcdce | Prompt: A black outline of a smartphone icon wit...
  ~ Prompt tokens: 35 | Tokens available for SVG: 5005
  ~ Generating 2 Candidates...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate failed: No geometric shapes found

In [ ]:
# Identify the remaining blank SVGs

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 18 blank SVGs.


The below process took 351 minutes to train with 18 samples.

In [20]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from the fixed submission
sub_df = pd.read_csv("./submission_fixed-3b-2048-p2.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 2  # We can be more aggressive here since it's our fourth pass
MAX_CONTEXT_WINDOW = 8192  # We increase this once more to handle cases beyond the 99th Token Count Percentile.

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found! Saving the previous submission as amplified.")
    sub_df.to_csv("./submission_amplified-3b-2048-p2.csv", index=False) 
else:
    # The 8192-Token Rescue Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # SEQUENTIAL GENERATION LOOP (Saves VRAM!)
        for i in range(NUM_CANDIDATES):
            print(f"  -> Generating Candidate {i+1}/{NUM_CANDIDATES}...")
            try:
                # THE KEY DIFFERENCE: max_new_tokens bumped to 8192 to prevent the guillotine
                raw_candidate = generator(
                    prompt_text, 
                    official_kaggle_grammar, 
                    do_sample=True, 
                    temperature=0.7, 
                    max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                    num_return_sequences=1, 
                )
                
                # Outlines might return a list of 1 string or just a string, handle both safely
                if isinstance(raw_candidate, list):
                    candidates = raw_candidate
                else:
                    candidates.append(raw_candidate)
                    
            except Exception as e:
                print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the select_best_svg_healed_updated function, which includes further improved healing and compliance checks before scoring.
        # (It already includes the is_kaggle_compliant_updated check and CLIP scoring)
        best_svg = select_best_svg_healed_updated(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the amplified file
        sub_df.to_csv("./submission_amplified-3b-2048-p2.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_amplified-3b-2048-p2.csv' is ready.")

Found 18 blank SVGs. Starting Rescue Run.

[1/18] Rescuing ID: af5c92c5dd805ba882bf2cf71f1c0da5 | Prompt: The image shows a black envelope icon wi...
  ~ Prompt tokens: 29 | Tokens available for SVG: 8153
  -> Generating Candidate 1/2...
  -> Generating Candidate 2/2...
  ~ Judging 2 successful candidates with SigLIP...
#2c2c2c" d="M182.64,163.18 L182.64,146.55 L175.99,146.55 L175.99,163.18 L182.64,163.18 Z"/><path fill="#2c2c2c" d="M189.29,163.18 L189.29,146.55 L182.64,146.55 L182.64,163.18 L189.29,163.18 Z"/></svg>

[2/18] Rescuing ID: 2efc5bf83782701ffe988db621c4bef9 | Prompt: A stylized logo featuring a geometric de...
  ~ Prompt tokens: 19 | Tokens available for SVG: 8163
  -> Generating Candidate 1/2...
  -> Generating Candidate 2/2...
  ~ Judging 2 successful candidates with SigLIP...
      [!] Candidate failed: No geometric shapes found.
      [!] Candidate 1 failed compliance. Length: 94
Raw string: <svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" height="200" wid

In [21]:
# Identify the blank SVGs from the amplified submission
missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs.")

Found 2 blank SVGs.


The below process took 307 minutes for 2 samples.

In [ ]:
import pandas as pd
import warnings
import torch
import gc 

# Silence warnings
warnings.filterwarnings("ignore", category=UserWarning, message=".*Error in LLMatcher.*")

# Identify the missing rows from the amplified submission
sub_df = pd.read_csv("./submission_amplified-3b-2048-p2.csv")
test_df = pd.read_csv("./dl-spring-2026-svg-generation/test.csv")

missing_mask = sub_df['svg'] == "<svg></svg>"
missing_ids = sub_df[missing_mask]['id'].tolist()
rescue_df = test_df[test_df['id'].isin(missing_ids)]

print(f"Found {len(rescue_df)} blank SVGs. Starting Rescue Run.")

NUM_CANDIDATES = 1  # We can be more aggressive here since it's our fifth pass
MAX_CONTEXT_WINDOW = 32768  # We can also try to push this up to the max token limit of our model to handle the extreme outlier cases.

# If there are no blanks, we can safely exit!
if len(rescue_df) == 0:
    print("No blanks found! Saving the previous submission as hallucination.")
    sub_df.to_csv("./submission_hallucination-3b-2048-p2.csv", index=False) 
else:
    # The 32768-Token Loop
    open_comment = "<!-" + "-"
    close_comment = "-" + "->"

    for count, pair in enumerate(rescue_df.iterrows()):
        idx, row = pair 
        print(f"\n[{count+1}/{len(rescue_df)}] Rescuing ID: {row['id']} | Prompt: {row['prompt'][:40]}...")
        
        # The Critical Prompt Injection 
        prompt_text = f"{open_comment} Description: {row['prompt']} {close_comment}\n```xml\n"
        candidates = [] 

        inputs = hf_tokenizer(prompt_text, return_tensors="pt")
        prompt_token_count = inputs["input_ids"].shape[1]
        
        # Calculate remaining tokens, minus a 10-token safety buffer
        safe_max_new_tokens = MAX_CONTEXT_WINDOW - prompt_token_count - 10
        print(f"  ~ Prompt tokens: {prompt_token_count} | Tokens available for SVG: {safe_max_new_tokens}")

        # SEQUENTIAL GENERATION LOOP (Saves VRAM!)
        for i in range(NUM_CANDIDATES):
            print(f"  -> Generating Candidate {i+1}/{NUM_CANDIDATES}...")
            try:
                # THE KEY DIFFERENCE: max_new_tokens bumped to 8192 to prevent the guillotine
                raw_candidate = generator(
                    prompt_text, 
                    official_kaggle_grammar, 
                    do_sample=True, 
                    temperature=0.7, 
                    max_new_tokens=safe_max_new_tokens,  # This is the rescue boost for truncated generations
                    num_return_sequences=1, 
                )
                
                # Outlines might return a list of 1 string or just a string, handle both safely
                if isinstance(raw_candidate, list):
                    candidates = raw_candidate
                else:
                    candidates.append(raw_candidate)
                    
            except Exception as e:
                print(f"    Generation failed for {NUM_CANDIDATES} candidates: {e}")
            
        # Crucial for 8GB GPUs: Clear out leftover KV cache after each candidate
        # Flush internal candidate cache
        gc.collect()
        torch.cuda.empty_cache() 
            
        print(f"  ~ Judging {len(candidates)} successful candidates with SigLIP...")
        
        # We use the select_best_svg_healed_updated function, which includes further improved healing and compliance checks before scoring.
        # (It already includes the is_kaggle_compliant_updated check and CLIP scoring)
        best_svg = select_best_svg_healed_updated(row["prompt"], candidates)
        print(best_svg[-200:])  # Print the last 200 chars of the best SVG for sanity check

        # Inject the rescued SVG back into the main dataframe in memory
        sub_df.loc[sub_df['id'] == row['id'], 'svg'] = best_svg
        
        # Save incrementally to the hallucination file
        sub_df.to_csv("./submission_hallucination-3b-2048-p2.csv", index=False)

        # Aggressive Memory Wiping for the Global Loop 
        # Explicitly delete the tokenizer tensor from this loop iteration
        if 'inputs' in locals():
            del inputs
            
        # Force Python to run Garbage Collection NOW
        gc.collect() 
        
        # NOW clear the CUDA cache (it will actually clear the fragmentation)
        torch.cuda.empty_cache()

    print("\nRescue complete! Final 'submission_hallucination-3b-2048-p2.csv' is ready.")

Found 2 blank SVGs. Starting Rescue Run.

[1/2] Rescuing ID: 2efc5bf83782701ffe988db621c4bef9 | Prompt: A stylized logo featuring a geometric de...
  ~ Prompt tokens: 19 | Tokens available for SVG: 32739
  -> Generating Candidate 1/1...
  ~ Judging 1 successful candidates with SigLIP...
5,162.5 C138.33,162.5 134.17,158.33 134.17,153.33 L134.17,149.17 L134.17,162.5 C134.17,167.5 138.33,171.67 142.5,171.67 L147.5,171.67 C151.67,171.67 155.83,167.5 155.83,162.5 L155.83,149.17 Z"/></svg>

[2/2] Rescuing ID: f6d1ad64-70d5-481f-b013-143d63450b5d | Prompt: An icon of an eye with a central pupil a...
  ~ Prompt tokens: 36 | Tokens available for SVG: 32722
  -> Generating Candidate 1/1...
  ~ Judging 1 successful candidates with SigLIP...
,10 L10,10 L10,14 C10,14.88 10.88,15.77 12,16.78 L12,16 L12,12 C12,10.88 10.88,9.79 10,9.79 L10,8 L14,8 ZM8,2 L8,4 C9.11,4 10,5.11 10,6.22 L10,10 L14,10 L14,6.22 C14,5.11 14.89,4 16,4 L16,2 Z"/></svg>

Rescue complete! Final 'submission_hallucination-3b-2048-p